# Audit kualitas anotasi gegenpressing

Notebook ini memeriksa integritas label dan membuat review queue untuk label
`Ragu` atau confidence rendah. Review queue sengaja tidak memuat outcome quick
regain agar peninjauan ulang tidak mengubah label perilaku berdasarkan hasil.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.annotation_audit import build_annotation_audit
from src.research_readiness import build_model_manifest

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'annotation_audit_current'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
annotations = pd.read_csv(
    PROCESSED_DIR / 'gegenpressing_annotation_manifest.csv'
)
episodes = pd.read_csv(PROCESSED_DIR / 'episode_manifest.csv')
model_manifest = build_model_manifest(episodes, annotations)

summary, review_queue, checks = build_annotation_audit(
    annotations,
    model_manifest,
)
summary.to_csv(OUTPUT_DIR / 'annotation_audit_summary.csv', index=False)
review_queue.to_csv(OUTPUT_DIR / 'annotation_review_queue.csv', index=False)
(OUTPUT_DIR / 'annotation_audit_checks.json').write_text(
    json.dumps({key: bool(value) for key, value in checks.items()}, indent=2),
    encoding='utf-8',
)

print('Quality checks:')
display(checks.to_frame())
print('Summary by match and split:')
display(summary)
print('Blinded review queue:', len(review_queue))
display(review_queue)
print('Artifacts:', OUTPUT_DIR)
if checks['all_annotations_completed']:
    print('Semua sampel sudah dianotasi; tinjau review queue sebelum training final.')
else:
    print('Audit sementara. Kembali ke notebook 08 untuk menyelesaikan queue.')

Quality checks:


,passed
all_annotations_completed,False
all_labels_valid,True
candidate_id_unique,True
confidence_complete,True
confidence_valid,True
uncertain_have_notes,True
annotator_complete,True


Summary by match and split:


,split,match_id,queue,completed,usable,positive,negative,uncertain,low_confidence
0,test,J03WPY,40,3,3,2,1,0,0
1,train,J03WMX,40,10,10,9,1,0,0
2,train,J03WN1,24,3,3,2,1,0,0
3,train,J03WOH,40,3,2,2,0,1,0
4,train,J03WOY,40,3,3,2,1,0,0
5,train,J03WR9,40,3,2,2,0,1,0
6,validation,J03WQQ,40,3,3,2,1,0,0


Blinded review queue: 2


,candidate_id,match_id,split,period_id,loss_timestamp,loss_frame,is_gegenpressing,annotation_confidence,annotation_notes,annotated_at_utc,annotator,review_reason
0,J03WOH_P2_F116224,J03WOH,train,2,0 days 00:10:48.960000,116224,-1.0,2.0,telat melakukan pressing,2026-10-01T07:51:14.951834+00:00,rizal,uncertain
1,J03WR9_P1_F70062,J03WR9,train,1,0 days 00:40:02.480000,70062,-1.0,2.0,"sepertinya tim merah sedang menguasai bola, te...",2026-10-01T07:52:31.792096+00:00,rizal,uncertain


Artifacts: C:\VAEP-Track\artifacts\annotation_audit_current
Audit sementara. Kembali ke notebook 08 untuk menyelesaikan queue.
